# Movie Industry Analysis - Phase 2 Group Project
**Group 5 Members:** Angela Mukami, [Member 2], [Member 3], [Member 4], [Member 5], [Member 6]  
**Date:** December 2025

---

## Project Goal
Analyze movie box office data to provide data-driven recommendations for what types of films a new studio should produce.

## Business Problem

Our company is creating a new movie studio but lacks expertise in film production. The head of the studio needs to understand:
- What types of films currently perform best at the box office?
- What characteristics make films commercially successful?
- How should the studio allocate production resources?

**Stakeholder:** Head of New Movie Studio  
**Objective:** Provide three actionable, data-driven recommendations for film production strategy

---
# SECTION 1: Data Loading and Preparation
**Contributor:** Angela Mukami  
**Branch:** `feature/data-loading`

---

In [1]:
# Import required libraries
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

### Load Box Office Mojo Data

In [ ]:
# TO BE COMPLETED BY ANGELA
# Load Box Office Mojo dataset
# Check for missing values

In [6]:
# Load Box Office Mojo data
bom_df = pd.read_csv('bom.movie_gross.csv')

# Quick overview
print(" Box Office Mojo Dataset Overview:")
print("="*60)
print(f"Shape: {bom_df.shape}")
print(f"\nColumns: {bom_df.columns.tolist()}")
print(f"\nFirst 5 rows:")
bom_df.head()

 Box Office Mojo Dataset Overview:
Shape: (3387, 5)

Columns: ['title', 'studio', 'domestic_gross', 'foreign_gross', 'year']

First 5 rows:


,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010


In [7]:
# Check year range
print("Year Range:")
print(f"Earliest year: {bom_df['year'].min()}")
print(f"Latest year: {bom_df['year'].max()}")
print(f"\nMovies per year:")
print(bom_df['year'].value_counts().sort_index())

print("\n" + "="*60)

# Check for missing data
print("\n Missing Values:")
print(bom_df.isnull().sum())

Year Range:
Earliest year: 2010
Latest year: 2018

Movies per year:
2010    328
2011    399
2012    400
2013    350
2014    395
2015    450
2016    436
2017    321
2018    308
Name: year, dtype: int64


 Missing Values:
title                0
studio               5
domestic_gross      28
foreign_gross     1350
year                 0
dtype: int64


Important decision: We'll focus on domestic_gross since it has much less missing data than foreign gross!

### Load IMDB Database

In [ ]:
# TO BE COMPLETED BY ANGELA
# Connect to IMDB SQLite database
# Load movie_basics and movie_ratings tables
# Filter for 2010-2018

In [2]:
# create connection to SQLite database
conn = sqlite3.connect('im.db')

# create cursor
cursor = conn.cursor()

In [3]:
# list all available tables
cursor.execute("""
SELECT name FROM sqlite_master 
WHERE type= 'table';
""")
cursor.fetchall()

[('movie_basics',),
 ('directors',),
 ('known_for',),
 ('movie_akas',),
 ('movie_ratings',),
 ('persons',),
 ('principals',),
 ('writers',)]

In [4]:
# Load just the first 5 rows to see the structure quickly
query1 = """SELECT * FROM movie_basics LIMIT 5;"""
df_movie_basics = pd.read_sql_query(query1, conn)
df_movie_basics.head()

,movie_id,primary_title,original_title,start_year,runtime_minutes,genres
0,tt0063540,Sunghursh,Sunghursh,2013,175.0,"Action,Crime,Drama"
1,tt0066787,One Day Before the Rainy Season,Ashad Ka Ek Din,2019,114.0,"Biography,Drama"
2,tt0069049,The Other Side of the Wind,The Other Side of the Wind,2018,122.0,Drama
3,tt0069204,Sabse Bada Sukh,Sabse Bada Sukh,2018,NaN,"Comedy,Drama"
4,tt0100275,The Wandering Soap Opera,La Telenovela Errante,2017,80.0,"Comedy,Drama,Fantasy"


In [5]:
# Load first 5 rows from movie_ratings
query2 = """SELECT * FROM movie_ratings LIMIT 5;"""
df_movie_ratings = pd.read_sql_query(query2, conn)
df_movie_ratings.head()

,movie_id,averagerating,numvotes
0,tt10356526,8.3,31
1,tt10384606,8.9,559
2,tt1042974,6.4,20
3,tt1043726,4.2,50352
4,tt1060240,6.5,21


Both tables have movie_id, which means we can merge them together!

What we learned about movie_ratings:

movie_id - Matches the movie_id in movie_basics

averagerating - IMDB rating (0-10 scale, like 8.3, 8.9, 6.4)

numvotes - Number of people who voted (31, 559, 50352)

Something to notice: Movie with 50,352 votes is more reliable than one with only 31 votes!

In [8]:
# now we can go back and load IMDB data for 2010-2018 with filtering criteria
# 
# FILTERING RATIONALE:
# 1. Years 2010-2018: Match the Box Office Mojo dataset timeframe for accurate merging
# 2. numvotes >= 1000: Ensure rating reliability by filtering out movies with too few votes
#    - Movies with <1000 votes may have skewed/unreliable ratings
#    - Industry standard: minimum sample size for statistical significance
# 3. runtime_minutes IS NOT NULL: Need runtime data for our analysis
# 4. genres != 'NaN': Need genre information to answer business question about film types

query_imdb = """
SELECT 
    mb.movie_id,
    mb.primary_title,
    mb.start_year,
    mb.runtime_minutes,
    mb.genres,
    mr.averagerating,
    mr.numvotes
FROM movie_basics mb
INNER JOIN movie_ratings mr ON mb.movie_id = mr.movie_id
WHERE mb.start_year >= 2010
    AND mb.start_year <= 2018
    AND mb.runtime_minutes IS NOT NULL
    AND mb.genres != 'NaN'
    AND mr.numvotes >= 1000
ORDER BY mr.numvotes DESC;
"""

print("Loading IMDB data for 2010-2018...")
imdb_df = pd.read_sql_query(query_imdb, conn)

print(f"Loaded {len(imdb_df):,} movies from IMDB")
print(f"\nFirst 10 most popular movies:")
imdb_df.head(10)

Loading IMDB data for 2010-2018...
Loaded 9,426 movies from IMDB

First 10 most popular movies:


,movie_id,primary_title,start_year,runtime_minutes,genres,averagerating,numvotes
0,tt1375666,Inception,2010,148.0,"Action,Adventure,Sci-Fi",8.8,1841066
1,tt1345836,The Dark Knight Rises,2012,164.0,"Action,Thriller",8.4,1387769
2,tt0816692,Interstellar,2014,169.0,"Adventure,Drama,Sci-Fi",8.6,1299334
3,tt1853728,Django Unchained,2012,165.0,"Drama,Western",8.4,1211405
4,tt0848228,The Avengers,2012,143.0,"Action,Adventure,Sci-Fi",8.1,1183655
5,tt0993846,The Wolf of Wall Street,2013,180.0,"Biography,Crime,Drama",8.2,1035358
6,tt1130884,Shutter Island,2010,138.0,"Mystery,Thriller",8.1,1005960
7,tt2015381,Guardians of the Galaxy,2014,121.0,"Action,Adventure,Comedy",8.1,948394
8,tt1431045,Deadpool,2016,108.0,"Action,Adventure,Comedy",8.0,820847
9,tt1392170,The Hunger Games,2012,142.0,"Action,Adventure,Sci-Fi",7.2,795227


### Merge Datasets

In [ ]:
# TO BE COMPLETED BY ANGELA
# Clean titles for matching
# Merge IMDB with Box Office Mojo on title and year
# Display merge statistics

### Why Merge IMDB with Box Office Mojo Data?

### Business Question
Our stakeholder (head of new movie studio) needs to know **what types of films** perform best at the box office.

### Data Strategy
To answer this question, we need **two types of information**:

1. **Box Office Mojo provides:**
   - Commercial success metrics (domestic & foreign gross revenue)
   - Which movies actually made money
   - Studio information

2. **IMDB provides:**
   - **Film characteristics** (genres, runtime)
   - **Quality indicators** (ratings, number of votes)
   - Movie metadata

### Why We Must Merge
**Neither dataset alone can answer our business question:**

- Box Office Mojo tells us **which movies succeeded financially** but doesn't tell us their **genres** or **characteristics**
- IMDB tells us **what type of films they are** but doesn't tell us their **box office performance**

**By merging these datasets**, we can identify patterns like:
- Which genres generate the highest revenue?
- Do highly-rated films perform better at the box office?
- What runtime is optimal for commercial success?
- Does quality (ratings) correlate with financial performance?

### Merge Strategy
We'll merge on **movie title AND year** because:
- Some movies share the same title 
- Matching on year ensures we get the correct movie
- Both datasets cover 2010-2018 timeframe

In [10]:
# Prepare both datasets for merging
bom_df['title_clean'] = bom_df['title'].str.lower().str.strip()
imdb_df['title_clean'] = imdb_df['primary_title'].str.lower().str.strip()
bom_df['year_int'] = bom_df['year'].astype(int)
imdb_df['year_int'] = imdb_df['start_year'].astype(int)

print("Data prepared for merging")
print(f"\nBox Office Mojo: {len(bom_df)} movies")
print(f"IMDB: {len(imdb_df)} movies")

Data prepared for merging

Box Office Mojo: 3387 movies
IMDB: 9426 movies


In [11]:
# Merge IMDB and Box Office Mojo data
# Merging on both title AND year to ensure we match the correct movies
# Using inner join to keep only movies present in BOTH datasets

movies_merged = pd.merge(
    bom_df,
    imdb_df,
    left_on=['title_clean', 'year_int'],
    right_on=['title_clean', 'year_int'],
    how='inner'
)

print(f"✅ Successfully merged datasets!")
print(f"\nOriginal datasets:")
print(f"   Box Office Mojo: {len(bom_df):,} movies")
print(f"   IMDB: {len(imdb_df):,} movies")
print(f"\nMerged dataset:")
print(f"   Combined: {len(movies_merged):,} movies")
print(f"   Match rate: {len(movies_merged)/len(bom_df)*100:.1f}% of Box Office movies")

print(f"\nColumns in merged dataset: {movies_merged.columns.tolist()}")
print(f"\nFirst 5 movies:")
movies_merged.head()

✅ Successfully merged datasets!

Original datasets:
   Box Office Mojo: 3,387 movies
   IMDB: 9,426 movies

Merged dataset:
   Combined: 1,778 movies
   Match rate: 52.5% of Box Office movies

Columns in merged dataset: ['title', 'studio', 'domestic_gross', 'foreign_gross', 'year', 'title_clean', 'year_int', 'movie_id', 'primary_title', 'start_year', 'runtime_minutes', 'genres', 'averagerating', 'numvotes']

First 5 movies:


,title,studio,domestic_gross,foreign_gross,year,title_clean,year_int,movie_id,primary_title,start_year,runtime_minutes,genres,averagerating,numvotes
0,Toy Story 3,BV,415000000.0,652000000,2010,toy story 3,2010,tt0435761,Toy Story 3,2010,103.0,"Adventure,Animation,Comedy",8.3,682218
1,Inception,WB,292600000.0,535700000,2010,inception,2010,tt1375666,Inception,2010,148.0,"Action,Adventure,Sci-Fi",8.8,1841066
2,Shrek Forever After,P/DW,238700000.0,513900000,2010,shrek forever after,2010,tt0892791,Shrek Forever After,2010,93.0,"Adventure,Animation,Comedy",6.3,167532
3,The Twilight Saga: Eclipse,Sum.,300500000.0,398000000,2010,the twilight saga: eclipse,2010,tt1325004,The Twilight Saga: Eclipse,2010,124.0,"Adventure,Drama,Fantasy",5.0,211733
4,Iron Man 2,Par.,312400000.0,311500000,2010,iron man 2,2010,tt1228705,Iron Man 2,2010,124.0,"Action,Adventure,Sci-Fi",7.0,657690


Complete Merged Dataset Columns:
From Box Office Mojo:

title - Original movie title from Box Office Mojo
studio - Production studio (BV, WB, etc.)
domestic_gross - US box office revenue
foreign_gross - International box office revenue
year - Release year

From IMDB:

movie_id - IMDB unique identifier
primary_title - Movie title from IMDB
start_year - Release year from IMDB
runtime_minutes - Movie length
genres - Comma-separated genres (e.g., "Action,Adventure,Sci-Fi")
averagerating - IMDB rating (0-10 scale)
numvotes - Number of IMDB user votes

Merge helper columns:

title_clean - Cleaned title for matching
year_int - Year as integer

In [12]:
# Check missing values in merged dataset
print(" Missing Values in Merged Dataset (1,778 movies, 2010-2018)")
print("="*60)
print(f"\nMissing values by column:")
print(movies_merged.isnull().sum())

print("\n" + "="*60)
print(f"\nMissing data percentages:")
missing_pct = (movies_merged.isnull().sum() / len(movies_merged) * 100).round(1)
print(missing_pct[missing_pct > 0])

print("\n" + "="*60)
print(f"\nForeign gross analysis:")
print(f"Total movies: {len(movies_merged):,}")
print(f"Movies WITH foreign_gross: {movies_merged['foreign_gross'].notna().sum():,}")
print(f"Movies WITHOUT foreign_gross: {movies_merged['foreign_gross'].isna().sum():,}")
print(f"Percentage missing: {movies_merged['foreign_gross'].isna().sum() / len(movies_merged) * 100:.1f}%")

 Missing Values in Merged Dataset (1,778 movies, 2010-2018)

Missing values by column:
title                0
studio               2
domestic_gross       9
foreign_gross      502
year                 0
title_clean          0
year_int             0
movie_id             0
primary_title        0
start_year           0
runtime_minutes      0
genres               0
averagerating        0
numvotes             0
dtype: int64


Missing data percentages:
studio             0.1
domestic_gross     0.5
foreign_gross     28.2
dtype: float64


Foreign gross analysis:
Total movies: 1,778
Movies WITH foreign_gross: 1,276
Movies WITHOUT foreign_gross: 502
Percentage missing: 28.2%


#### Data Quality Assessment: Revenue Metrics

**Missing Data Analysis:**
- `domestic_gross`: 9 missing (0.5%) - **Excellent**
- `foreign_gross`: 502 missing (28.2%) - **Significant gaps**

**Decision: Use Domestic Gross Only**

We will focus our analysis on domestic box office performance because:
1. **Data completeness**: 99.5% of movies have domestic gross data
2. **Reliability**: Domestic data is consistently reported
3. **Business relevance**: US market remains the largest single film market globally
4. **Statistical validity**: Missing 28% of foreign data would bias our analysis

This decision provides more reliable insights for our stakeholder's investment decisions.

## Analysis Strategy

Our analysis will answer three key business questions to guide film production decisions:

### Analysis 1: Genre Performance
**Question:** Which genres generate the highest box office revenue?
- Metric: Average domestic gross by primary genre
- Filter: Genres with ≥20 movies for statistical reliability
- Output: Top-performing genres ranked by revenue

### Analysis 2: Runtime Optimization  
**Question:** What is the optimal movie length for box office success?
- Metric: Average domestic gross by runtime category
- Categories: Short (<90min), Standard (90-120min), Long (120-150min), Very Long (>150min)
- Output: Ideal runtime range for commercial success

### Analysis 3: Genre-Runtime Interaction
**Question:** What is the optimal runtime for each top-performing genre?
- Metric: Average domestic gross by genre AND runtime category
- Output: Specific runtime recommendations per genre
- Value: Most actionable insight - combines "what to make" with "how long it should be"

**Why not analyze ratings?**
We deliberately exclude IMDB ratings from our primary analysis because:
- Ratings are determined POST-release (not predictable during production planning)
- Ratings depend on audience votes (outside studio control)
- A studio cannot decide "we will make an 8.5-rated film" before production
- Our stakeholder needs actionable decisions they can control: genre choice and runtime

---
# SECTION 2: Analysis 1 - Genre Performance
**Contributor:** [Member 2 Name]  
**Branch:** `feature/genre-analysis`

**Task:** Analyze which genres generate the highest box office revenue

---

### Extract Primary Genres

In [ ]:
# TO BE COMPLETED BY PERSON 2
# Look at the genre distribution
# Check how many unique genre combinations exist
# Extract primary genre 
# Filter for genres with >= 20 movies

### Calculate Genre Performance Metrics

In [ ]:
# TO BE COMPLETED BY PERSON 2
# Now let's see which genres make the most money
# Calculate count, average and median domestic gross by genre
# Sort by average revenue
# Display results table

### Visualization 1: Genre Performance

In [ ]:
# TO BE COMPLETED BY PERSON 2
# Create horizontal bar chart
# Show top genres by average revenue
# Color-code by performance level
# Save as 'genre_performance.png'

---
# SECTION 3: Analysis 2 - Runtime Impact
**Contributor:** [Member 3 Name]  
**Branch:** `feature/runtime-analysis`

**Task:** Analyze how movie length affects box office performance

---

### Create Runtime Categories

In [ ]:
# TO BE COMPLETED BY PERSON 3
# Create runtime categories: Short (<90), Standard (90-120), Long (120-150), Very Long (>150)
# Display distribution of movies across categories

### Calculate Runtime Performance

In [ ]:
# TO BE COMPLETED BY PERSON 3
# Calculate average domestic gross by runtime category
# Display results

### Visualization 2: Runtime Performance

In [ ]:
# TO BE COMPLETED BY PERSON 3
# Create bar chart showing runtime categories vs revenue
# Color-code by category
# Save as 'runtime_performance.png'

---
# SECTION 4: Analysis 3 - Genre-Runtime Interaction
**Contributor:** [Member 4 Name]  
**Branch:** `feature/genre-runtime-interaction`

**Task:** Analyze optimal runtime for each top-performing genre

---

### Analyze Genre-Runtime Combinations

In [ ]:
# TO BE COMPLETED BY PERSON 4
# Focus on top 5 genres from Analysis 1
# Calculate average revenue by genre AND runtime category
# Display interaction results

### Visualization 3: Genre-Runtime Interaction

In [ ]:
# TO BE COMPLETED BY PERSON 4
# Create grouped bar chart or heatmap
# Show optimal runtime for each genre
# Save as 'genre_runtime_comparison.png'

---
# SECTION 5: Summary of Findings and Recommendations
**Contributor:** [Member 5 Name]  
**Branch:** `feature/findings-recommendations`

**Task:** Synthesize findings and create business recommendations

---

## Summary of Key Findings

TO BE COMPLETED BY PERSON 5:
- Write Finding 1: Genre Performance (based on Analysis 1)
- Write Finding 2: Runtime Impact (based on Analysis 2)
- Write Finding 3: Optimal Runtime by Genre (based on Analysis 3)

## Business Recommendations

TO BE COMPLETED BY PERSON 5:
- Write Recommendation 1: Genre focus
- Write Recommendation 2: Runtime strategy
- Write Recommendation 3: What to avoid

---
# SECTION 6: Documentation
**Contributor:** [Member 6 Name]  
**Branch:** `feature/documentation`

**Task:** Create README.md and presentation

**Note:** This will be done in separate files, not in the notebook

---